# Gaussian Mixture Model in JAX

This notebook demonstrates fitting a simple Gaussian Mixture Model using Expectation-Maximization in JAX.

The example fits 3 Gaussians to synthetic data.

In [ ]:
import jax
import jax.numpy as jnp
from jax.scipy.stats import multivariate_normal
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(11)
centers = jnp.array([[-4.0,-2.0],[0.0,4.0],[4.0,-1.0]])
samples = []
for i, c in enumerate(centers):
    key, sub = jax.random.split(key)
    samples.append(jax.random.normal(sub, (120,2)) * 0.8 + c)
X = jnp.concatenate(samples, axis=0)
X = X[jax.random.permutation(key, X.shape[0])]

In [ ]:
def initialize_params(k, dim, key):
    keys = jax.random.split(key, 3)
    weights = jnp.ones((k,)) / k
    means = X[jax.random.choice(keys[0], X.shape[0], (k,), replace=False)]
    covs = jnp.stack([jnp.eye(dim) for _ in range(k)])
    return weights, means, covs

def e_step(X, weights, means, covs):
    k = weights.shape[0]
    probs = jnp.stack([weights[i] * multivariate_normal.pdf(X, means[i], covs[i]) for i in range(k)], axis=1)
    responsibilities = probs / jnp.sum(probs, axis=1, keepdims=True)
    return responsibilities

def m_step(X, responsibilities):
    Nk = jnp.sum(responsibilities, axis=0)
    weights = Nk / X.shape[0]
    means = (responsibilities.T @ X) / Nk[:, None]
    covs = []
    for i in range(responsibilities.shape[1]):
        diff = X - means[i]
        cov = (responsibilities[:, i][:, None] * diff).T @ diff / Nk[i]
        covs.append(cov + 1e-6 * jnp.eye(X.shape[1]))
    covs = jnp.stack(covs)
    return weights, means, covs

def fit_gmm(X, k=3, steps=30, key=jax.random.PRNGKey(0)):
    weights, means, covs = initialize_params(k, X.shape[1], key)
    for _ in range(steps):
        responsibilities = e_step(X, weights, means, covs)
        weights, means, covs = m_step(X, responsibilities)
    return weights, means, covs, responsibilities

weights, means, covs, responsibilities = fit_gmm(X, key=key)
assignments = jnp.argmax(responsibilities, axis=1)

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(X[:,0], X[:,1], c=assignments, cmap='viridis')
plt.scatter(means[:,0], means[:,1], c='red', marker='x', s=80)
plt.title('GMM clustering')
plt.show()